In [ ]:
%pip install pypdf python-docx pytesseract pillow

In [ ]:
import os
import sys

os.environ.pop("SPARK_HOME", None)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession, functions as F
from pypdf import PdfReader
from docx import Document
from PIL import Image
from io import BytesIO
import re

jdbc_jar_path = "/Users/jatin/Library/DBeaverData/drivers/maven/maven-central/org.postgresql/postgresql-42.7.2.jar"

spark = (
    SparkSession.builder
    .appName("intelligent-text-extraction")
    .master("local[*]")
    .config("spark.jars", jdbc_jar_path)
    .getOrCreate()
)

pg_url = "jdbc:postgresql://localhost:5432/Project-1"
pg_properties = {
    "user": "postgres",
    "password": "pass",
    "driver": "org.postgresql.Driver",
}
metadata_table = '"CDAC".file_metadata'

In [ ]:
input_path = "../raw"

In [ ]:
def extract_text(file_path, file_extension):
    """
    Generic document text extraction router.

    Routes the file to the appropriate extraction
    method based on its file extension.
    """

    extension = file_extension.lower().replace(".", "")

    if extension == "pdf":
        return extract_text_from_pdf(file_path), "PDF_TEXT_EXTRACTION"

    elif extension == "docx":
        return extract_text_from_docx(file_path), "DOCX_TEXT_EXTRACTION"

    elif extension in ["jpg", "jpeg", "png"]:
        return extract_text_from_image(file_path), "OCR"

    else:
        return None, "UNSUPPORTED_FORMAT"

def extract_text_from_image(file_path):
    """
    OCR function for image-based documents.

    This function is intentionally isolated so that
    the OCR engine can be replaced later without
    changing the rest of the pipeline.
    """

    try:
        with open(file_path, "rb") as f:
            file_data = f.read()

        image = Image.open(BytesIO(file_data))

        # OCR implementation will be added here.
        # Keep this function isolated from the rest of the pipeline.

        return None

    except Exception as e:
        raise Exception(f"Image OCR failed: {str(e)}")

def extract_text_from_docx(file_path):
    """
    Extract text from a DOCX document.
    """

    try:
        with open(file_path, "rb") as f:
            file_data = f.read()

        document = Document(BytesIO(file_data))

        paragraphs = []

        for paragraph in document.paragraphs:
            if paragraph.text.strip():
                paragraphs.append(paragraph.text.strip())

        return "\n".join(paragraphs).strip()

    except Exception as e:
        raise Exception(f"DOCX extraction failed: {str(e)}")

def extract_text_from_pdf(file_path):
    """
    Extract text from a text-based PDF.
    Returns extracted text as a string.
    """

    try:
        with open(file_path, "rb") as f:
            pdf_data = f.read()

        reader = PdfReader(BytesIO(pdf_data))

        pages_text = []

        for page in reader.pages:
            text = page.extract_text()

            if text:
                pages_text.append(text)

        return "\n".join(pages_text).strip()

    except Exception as e:
        raise Exception(f"PDF extraction failed: {str(e)}")

In [ ]:
metadata_df = spark.read.jdbc(url=pg_url, table=metadata_table, properties=pg_properties)

metadata_df.toPandas()

In [ ]:
metadata_df = metadata_df.filter(
    F.col("status") == "NEW"
)

metadata_df.toPandas()

In [ ]:
results = []

for row in metadata_df.collect():

    file_id = row["file_id"]
    file_name = row["file_name"]
    file_path = row["file_path"]
    file_extension = row["file_extension"]

    try:

        raw_text, extraction_method = extract_text(
            file_path,
            file_extension
        )

        extraction_status = "SUCCESS"
        error_message = None

    except Exception as e:

        raw_text = None
        extraction_method = None
        extraction_status = "FAILED"
        error_message = str(e)

    results.append(
        (
            file_id,
            file_name,
            file_path,
            file_extension,
            raw_text,
            extraction_method,
            extraction_status,
            error_message
        )
    )

In [ ]:
extracted_schema = """
file_id STRING,
file_name STRING,
file_path STRING,
file_extension STRING,
raw_text STRING,
extraction_method STRING,
extraction_status STRING,
error_message STRING
"""

extracted_df = spark.createDataFrame(
    results,
    schema=extracted_schema
)

extracted_df.toPandas()

In [ ]:
extracted_df = (
    extracted_df
    .withColumn(
        "text_length",
        F.length(F.col("raw_text"))
    )
    .withColumn(
        "needs_ocr",
        F.when(
            (F.col("file_extension") == "pdf") &
            (
                F.col("text_length").isNull() |
                (F.col("text_length") < 50)
            ),
            True
        ).otherwise(False)
    )
)

extracted_df.toPandas()